In [ ]:
import pandas as pd
import nltk
from collections import Counter
import re

# Download the specific latest versions of NLTK data
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')

def extract_landmarks(file_path, text_column='instruction'):
    # 1. Load the data
    if file_path.endswith('.parquet'):
        df = pd.read_parquet(file_path)
    else:
        df = pd.read_csv(file_path)

    print(f"Loaded {len(df)} rows. Analyzing nouns...")

    all_nouns = []
    
    # 2. Process each instruction
    for text in df[text_column].dropna():
        # Clean text
        text = re.sub(r'[^a-zA-Z\s]', '', str(text).lower())
        
        # Tokenize and Tag parts of speech
        tokens = nltk.word_tokenize(text)
        # Use the specific tagger engine
        tagged = nltk.pos_tag(tokens, tagset=None, lang='eng')
        
        # Filter for Nouns (NN = Singular, NNS = Plural)
        exclude = {'meters', 'feet', 'blocks', 'turn', 'left', 'right', 'straight', 'way', 'distance', 'steps'}
        nouns = [word for word, pos in tagged if pos.startswith('NN') and word not in exclude]
        
        all_nouns.extend(nouns)

    # 3. Count frequencies
    counts = Counter(all_nouns)
    
    print("\n--- Top 30 Potential Landmarks in Manhattan Data ---")
    print(f"{'WORD':15} | {'FREQUENCY'}")
    print("-" * 30)
    for word, freq in counts.most_common(30):
        print(f"{word:15} | {freq}")

# --- EXECUTION ---
# Make sure the path to your file is correct!
# extract_landmarks('data/your_manhattan_file.csv')